# TalentIQ — Exploratory Data Analysis
### Phase 2 | Binary Classification | Resume Screening Pipeline

---

## What are we trying to solve?

Companies receive **thousands of resumes** for every job posting. Going through each one manually is slow, inconsistent, and expensive. A recruiter might spend 6–10 seconds on a resume and still miss great candidates — or waste time interviewing wrong ones.

**TalentIQ solves this** by building a machine learning model that looks at a candidate's resume data — their CGPA, skills, experience, projects, certifications — and predicts whether they should be hired or not. This is a **binary classification problem**: the output is either `1 (Hired)` or `0 (Not Hired)`.

---

## What is EDA and why do we do it first?

Before we build any model, we need to **understand the data**. EDA (Exploratory Data Analysis) is that step.

Think of it like this — if you're cooking a new dish, you don't just throw ingredients into a pan without checking them first. You check if anything is expired, if you have enough of each item, if some ingredients need to be prepped differently. EDA is exactly that for a dataset.

In this notebook we will:
- Look at the shape and structure of the data
- Check if the classes (Hired / Not Hired) are balanced
- Find missing values
- Detect outliers
- See which features are most correlated with getting hired
- Understand categorical features like Education Level and University Tier

**Everything we find here directly decides what we do in preprocessing.**

---

## File Map — Where each problem gets fixed

| What we find in EDA | Where we fix it |
|---|---|
| Missing values | `src/preprocessing.py` — impute_missing() |
| Outliers | `src/preprocessing.py` — cap_outliers() |
| Categorical columns need encoding | `src/preprocessing.py` — ordinal_encode(), onehot_encode() |
| Class imbalance | `src/train.py` — SMOTE applied on training set |
| Feature relationships with target | `src/feature_engineering.py` — used to design 7 engineered features |

---

## Step 0 — Import Libraries

We load everything we need here. Nothing special — pandas for data, seaborn and matplotlib for plots, numpy for math, yaml to read our config file.

In [ ]:
import sys
sys.path.append('..')  # lets us import from src/ folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

# Clean plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

print('All libraries loaded successfully.')

---
## Step 1 — Load the Dataset

### Problem
Our dataset has **200,000 candidates**. Running the full dataset every time during development is slow — it takes more time, uses more memory, and makes testing painful. We don't want to wait 2 minutes just to check if a plot works.

### Solution
We built a **mode system** in our config. If `mode = sample`, we load only 5000 rows — fast enough to test everything instantly. If `mode = full`, we load all 200K rows for actual training.

### Where is this implemented?
In `src/config_loader.py` → `load_data()` function. It reads the `mode` from `config/config.yaml` and loads accordingly. You just change one word in the YAML file and the whole pipeline switches.

In [ ]:
# Read config to know which mode we're in
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

mode        = cfg.get('mode', 'full')
sample_size = cfg.get('sample_size', 5000)
raw_path    = '../' + cfg['paths']['raw_data']

print(f'Current mode : {mode.upper()}')
print(f'Dataset path : {raw_path}')

if mode == 'sample':
    df = pd.read_csv(raw_path, nrows=sample_size)
    print(f'Loaded {len(df)} rows (SAMPLE mode)')
else:
    df = pd.read_csv(raw_path)
    print(f'Loaded {len(df)} rows (FULL mode)')

---
## Step 2 — Basic Dataset Info

### Problem
We just loaded the data. We have no idea what columns exist, what data types they are, or what the values look like. Before doing anything, we need a quick overview.

### What we check
- Shape (rows × columns)
- Data types — are numerical columns actually stored as numbers? Or as strings?
- First few rows — does the data look like what we expect?
- Basic statistics — min, max, mean, std for each column

### Why this matters
Sometimes a column like `CGPA` might be stored as a string (object type) if there are entries like `"8.5/10"`. We'd catch that here and handle it in preprocessing. Also, if max CGPA is 100 when it should be 10, that's a red flag we catch here.

In [ ]:
print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
print('=== Column Data Types ===')
print(df.dtypes)
print()
print('=== First 5 Rows ===')
df.head()

In [ ]:
print('=== Descriptive Statistics ===')
print('(Look for anything suspicious — values too high/low, weird min/max)')
df.describe().T.round(2)

---
## Step 3 — Class Distribution & Imbalance Check

### Problem
This is one of the most important checks in a classification project. Imagine if out of 200,000 candidates, only 2,000 were hired (1%) and 198,000 were not (99%). If we train a model on this, it could just predict "Not Hired" for everyone and still be 99% accurate — but it's completely useless.

This is called **class imbalance**, and it ruins models if not handled properly.

### Solution
We calculate the **imbalance ratio** = (majority class count) ÷ (minority class count). If this ratio is greater than 3.0, we apply **SMOTE** — a technique that creates synthetic samples of the minority class to balance things out.

### Where is this implemented?
- The check is done here in EDA
- The fix (SMOTE) is applied in `src/train.py` — and only on the **training set** to avoid data leakage
- The SMOTE setting is controlled by `config/config.yaml → smote: apply: true`

In [ ]:
class_counts = df['Hired'].value_counts()
class_pct    = df['Hired'].value_counts(normalize=True) * 100

print('=== Class Distribution ===')
print(f'  Not Hired (0) : {class_counts[0]:>6,}  ({class_pct[0]:.1f}%)')
print(f'  Hired     (1) : {class_counts[1]:>6,}  ({class_pct[1]:.1f}%)')

imbalance_ratio = class_counts[0] / class_counts[1]
print(f'\n  Imbalance Ratio : {imbalance_ratio:.2f}x')

if imbalance_ratio > 3.0:
    print('  ⚠️  Ratio > 3.0 → SMOTE will be applied in Phase 3 training')
    print('  ⚠️  Do NOT use accuracy as the primary metric — use F1-macro')
else:
    print('  ✅  Ratio ≤ 3.0 → Imbalance is manageable')

In [ ]:
import os
os.makedirs('../reports/figures', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(
    ['Not Hired (0)', 'Hired (1)'],
    class_counts.values,
    color=['#EF5350', '#42A5F5'],
    edgecolor='white', linewidth=1.5, width=0.5
)
axes[0].set_title('Class Distribution — Raw Count', fontweight='bold')
axes[0].set_ylabel('Number of Candidates')
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=11)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=['Not Hired', 'Hired'],
    autopct='%1.1f%%',
    colors=['#EF5350', '#42A5F5'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Distribution — Percentage', fontweight='bold')

plt.suptitle(f'Target Variable: Hired  |  Imbalance Ratio = {imbalance_ratio:.2f}x',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → reports/figures/class_distribution.png')

---
## Step 4 — Missing Values Check

### Problem
Real-world datasets are messy. Candidates might not have filled in every field. Some rows might have blank values for CGPA, or missing information about certifications. If we pass missing values directly into a model, it will either crash or give wrong predictions.

### Solution
We first **visualize** where missing values are. Then in preprocessing, we fill them in:
- **Numerical columns** (like CGPA, SkillsScore) → filled with the **median** value. We use median instead of mean because it's not affected by extreme outliers.
- **Categorical columns** (like EducationLevel) → filled with the **mode** (most common value).

### Where is this implemented?
`src/preprocessing.py` → `impute_missing()` function. It reads which columns are numerical and which are categorical from `config/features.yaml` and applies the right strategy to each.

In [ ]:
missing       = df.isnull().sum()
missing_pct   = (missing / len(df) * 100).round(2)
missing_df    = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df    = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

total_missing = df.isnull().sum().sum()
print(f'Total missing cells across dataset: {total_missing}')

if missing_df.empty:
    print('\n✅  No missing values found. Dataset is complete.')
    print('    Nothing to impute in preprocessing for missing values.')
else:
    print(f'\n⚠️  {len(missing_df)} column(s) have missing values:')
    print(missing_df)
    print('\n→ Fix: impute_missing() in src/preprocessing.py')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    df.isnull(),
    yticklabels=False,
    cbar=False,
    cmap='viridis',
    ax=ax
)
ax.set_title(
    'Missing Values Heatmap\n(Yellow = missing cell | Purple = present)',
    fontweight='bold'
)
ax.set_xlabel('Columns')
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150)
plt.show()
print('Plot saved → reports/figures/missing_values.png')

---
## Step 5 — Outlier Detection (IQR Method)

### Problem
Outliers are extreme values that don't represent reality. For example, if a candidate's CGPA is listed as 9.8 out of 10 but one row has 47.5 — that's clearly a data entry error. Or someone claims 50 years of work experience when the scale goes from 0 to 10. These extreme values can **distort a model's understanding** of the data and push predictions in the wrong direction.

### Solution — IQR Capping
We use the **IQR (Interquartile Range) method** to detect outliers:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1
- Any value below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR` is an outlier

We don't delete these rows — we **cap** them. Values too low get set to the lower bound, values too high get set to the upper bound. This keeps the data while removing the extreme effect.

### Where is this implemented?
`src/preprocessing.py` → `cap_outliers()` function. The columns to check are defined in `config/features.yaml → outlier_columns`.

In [ ]:
outlier_cols = ['CGPA', 'YearsExperience', 'SkillsScore', 'SoftSkillsScore']
outlier_cols = [c for c in outlier_cols if c in df.columns]

print('=== Outlier Summary — IQR Method ===')
print(f'{"Column":<22} {"Q1":>7} {"Q3":>7} {"IQR":>7} {"Lower Bound":>13} {"Upper Bound":>13} {"Outlier Count":>14}')
print('-' * 90)

outlier_summary = {}
for col in outlier_cols:
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = {'lower': lower, 'upper': upper, 'count': count}
    print(f'{col:<22} {Q1:>7.2f} {Q3:>7.2f} {IQR:>7.2f} {lower:>13.2f} {upper:>13.2f} {count:>14}')

print()
print('→ Fix: cap_outliers() in src/preprocessing.py will clip these values to bounds')

In [ ]:
fig, axes = plt.subplots(1, len(outlier_cols), figsize=(15, 5))

for ax, col in zip(axes, outlier_cols):
    bp = ax.boxplot(
        df[col].dropna(),
        patch_artist=True,
        boxprops=dict(facecolor='#90CAF9', color='#1565C0'),
        medianprops=dict(color='#B71C1C', linewidth=2.5),
        whiskerprops=dict(color='#1565C0', linewidth=1.5),
        capprops=dict(color='#1565C0', linewidth=1.5),
        flierprops=dict(marker='o', markerfacecolor='#EF5350',
                        markeredgecolor='#B71C1C', markersize=4, alpha=0.5)
    )
    n_out = outlier_summary[col]['count']
    ax.set_title(f'{col}\n({n_out} outliers)', fontweight='bold', fontsize=10)
    ax.set_xticks([])

plt.suptitle('Boxplots — Outlier Detection (Red dots = outliers to be capped)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/outlier_boxplots.png', dpi=150)
plt.show()
print('Plot saved → reports/figures/outlier_boxplots.png')

---
## Step 6 — Correlation Heatmap

### Problem
We have many numerical features. We need to know **which features are actually related to getting hired**, and which ones are basically noise. Also, if two features are extremely correlated with each other (like CGPA and SkillsScore), they might be redundant — giving the model the same information twice.

### Solution — Pearson Correlation
We calculate the **Pearson correlation** between every pair of numerical features. Values close to +1 or -1 mean strong relationship. Values close to 0 mean no linear relationship.

We specifically look at the `Hired` column to find the **top 5 predictors** — the features most strongly related to whether someone gets hired.

### Where is this used?
These findings directly inform `src/feature_engineering.py`. When we design the weighted formulas for EmployabilityScore, PortfolioStrength etc., we give **higher weights to features with stronger correlation to Hired**.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr     = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask    = np.triu(np.ones_like(corr, dtype=bool))  # only show lower triangle

sns.heatmap(
    corr, mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 8},
    ax=ax
)
ax.set_title(
    'Pearson Correlation Heatmap — All Numerical Features\n'
    '(Red = positive correlation | Blue = negative correlation)',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → reports/figures/correlation_heatmap.png')

In [ ]:
if 'Hired' in corr.columns:
    top5 = corr['Hired'].drop('Hired').abs().sort_values(ascending=False).head(5)

    print('=== Top 5 Features Most Correlated With Getting Hired ===')
    print('(Higher |r| = stronger relationship with Hired)')
    print()
    for i, (feat, val) in enumerate(top5.items(), 1):
        direction = '+' if corr.loc[feat, 'Hired'] > 0 else '-'
        print(f'  #{i}  {feat:<25}  |r| = {val:.4f}  (direction: {direction})')

    print()
    print('→ These features get higher weights in feature_engineering.py formulas')

---
## Step 7 — Feature vs Class Boxplots ★

### Problem
The correlation heatmap tells us numbers. But we need to **visually confirm** that there's actually a visible difference in feature values between hired and not-hired candidates. If the boxplots for Hired=0 and Hired=1 completely overlap for a feature, that feature probably doesn't help the model much.

### What we're looking for
For a feature to be useful, the two boxplots (one for each class) should be **separated** — the medians should be noticeably different. For example, if hired candidates consistently have higher CGPA, the blue box (Hired=1) should sit higher than the red box (Hired=0).

### Where is this used?
This is a visual validation step. It confirms our feature engineering decisions in `src/feature_engineering.py`. If a feature shows no separation here, we might drop it or reconsider its weight in our engineered features.

In [ ]:
box_features = [
    'CGPA', 'SkillsScore', 'SoftSkillsScore', 'YearsExperience',
    'Projects', 'Hackathons', 'Certifications', 'Internships'
]
box_features = [c for c in box_features if c in df.columns]

n_cols   = 4
n_rows   = (len(box_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes      = axes.flatten()

palette = {0: '#EF5350', 1: '#42A5F5'}

for i, col in enumerate(box_features):
    sns.boxplot(
        data=df, x='Hired', y=col,
        palette=palette,
        width=0.45, linewidth=1.5,
        flierprops=dict(marker='o', markersize=3, alpha=0.3),
        ax=axes[i]
    )
    axes[i].set_title(col, fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Hired  (0 = Not Hired | 1 = Hired)')
    axes[i].set_ylabel('Value')

    # Show median values on the plot
    for cls in [0, 1]:
        med = df[df['Hired'] == cls][col].median()
        axes[i].text(cls, med, f' {med:.1f}', va='bottom',
                     fontsize=9, color='black', fontweight='bold')

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Feature vs Class Boxplots\n'
    'Red = Not Hired (0)  |  Blue = Hired (1)  |  Wider separation = more useful feature',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('../reports/figures/feature_vs_class_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → reports/figures/feature_vs_class_boxplots.png')

---
## Step 8 — Categorical Feature Analysis

### Problem
Some of our features are not numbers — they're categories. Things like `EducationLevel` (Bachelor, Master, PhD), `UniversityTier` (Tier 1, 2, 3), and `CompanyType`. ML models can't work with text directly. We need to:
1. First **understand** if these categories actually impact hiring chances
2. Then **encode** them into numbers in preprocessing

### What we check here
We calculate the **hiring rate** for each category — out of all candidates in that category, what % actually got hired? If PhD candidates have an 80% hire rate but High School has 20%, that's a significant signal the model needs to pick up.

### Where is this implemented?
- **Ordinal encoding** (for EducationLevel, UniversityTier — they have a natural order) → `src/preprocessing.py → ordinal_encode()`. The mapping (High School=1, Bachelor=2, Master=3, PhD=4) is stored in `config/features.yaml → ordinal_maps`.
- **One-hot encoding** (for CompanyType, ProgrammingLanguages — no natural order) → `src/preprocessing.py → onehot_encode()`.

In [ ]:
cat_cols = [c for c in ['EducationLevel', 'UniversityTier', 'CompanyType'] if c in df.columns]

for col in cat_cols:
    grp = df.groupby(col)['Hired'].agg(['count', 'sum', 'mean'])
    grp.columns = ['Total Candidates', 'Hired Count', 'Hire Rate']
    grp['Hire Rate'] = (grp['Hire Rate'] * 100).round(1)
    grp = grp.sort_values('Hire Rate', ascending=False)

    print(f'\n=== {col} ===')
    print(grp.to_string())
    print()

In [ ]:
n = len(cat_cols)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))

if n == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    hire_rate = df.groupby(col)['Hired'].mean().sort_values(ascending=False) * 100
    colors    = sns.color_palette('Blues_d', len(hire_rate))

    bars = ax.bar(
        hire_rate.index, hire_rate.values,
        color=colors, edgecolor='white', linewidth=1.2
    )
    ax.set_title(f'Hire Rate by {col}', fontweight='bold')
    ax.set_ylabel('Hire Rate (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=25)

    for bar, val in zip(bars, hire_rate.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5,
            f'{val:.1f}%',
            ha='center', fontsize=10, fontweight='bold'
        )

plt.suptitle(
    'Categorical Features — Hiring Rate per Category\n'
    '(Higher % = that category gets hired more often)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('../reports/figures/categorical_hire_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → reports/figures/categorical_hire_rates.png')

---
## Step 9 — EDA Summary & Key Findings

This is the most important cell. After all the analysis above, we document what we found and what action each finding triggers in the pipeline. Think of this as the handoff note from EDA to preprocessing.

In [ ]:
print('=' * 60)
print('  TALENTIQ — EDA KEY FINDINGS & ACTIONS')
print('=' * 60)

print(f'\n📦  DATASET')
print(f'    Rows loaded  : {df.shape[0]:,}')
print(f'    Columns      : {df.shape[1]}')
print(f'    Mode         : {mode.upper()}')

print(f'\n🎯  CLASS BALANCE')
print(f'    Not Hired : {class_counts[0]:,}  ({class_pct[0]:.1f}%)')
print(f'    Hired     : {class_counts[1]:,}  ({class_pct[1]:.1f}%)')
print(f'    Ratio     : {imbalance_ratio:.2f}x')
if imbalance_ratio > 3.0:
    print(f'    Action    : Apply SMOTE in src/train.py (on train set only)')
    print(f'    Metric    : Use F1-macro, NOT accuracy')
else:
    print(f'    Action    : No SMOTE needed. Monitor F1-macro anyway.')

print(f'\n❓  MISSING VALUES')
total_missing = df.isnull().sum().sum()
if total_missing == 0:
    print(f'    None found. No imputation needed.')
else:
    print(f'    Total missing: {total_missing}')
    print(f'    Action: impute_missing() in src/preprocessing.py')
    print(f'    Strategy: median for numerical, mode for categorical')

print(f'\n📊  TOP PREDICTORS')
if 'Hired' in corr.columns:
    for i, (feat, val) in enumerate(top5.items(), 1):
        print(f'    #{i}  {feat:<25}  |r| = {val:.4f}')

print(f'\n🔧  OUTLIERS')
for col, info in outlier_summary.items():
    print(f'    {col:<22} → {info["count"]} outliers → will be capped in preprocessing')

print(f'\n🏷️   ENCODING PLAN')
print(f'    EducationLevel  → Ordinal encode (HS=1, Bach=2, Master=3, PhD=4)')
print(f'    UniversityTier  → Ordinal encode (Tier3=1, Tier2=2, Tier1=3)')
print(f'    CompanyType     → One-hot encode (no natural order)')
print(f'    ProgrammingLang → One-hot encode (no natural order)')

print(f'\n📁  PLOTS SAVED')
plots = [
    'class_distribution.png',
    'missing_values.png',
    'outlier_boxplots.png',
    'correlation_heatmap.png',
    'feature_vs_class_boxplots.png',
    'categorical_hire_rates.png'
]
for p in plots:
    print(f'    reports/figures/{p}')

print(f'\n✅  EDA COMPLETE — Ready to proceed to preprocessing')
print('=' * 60)